## Compiler Optimization

In [1]:
import torch
import os
from Simulator.simulator import TOGSimulator

fusion_config = "/workspace/PyTorchSim/tutorial/session1/togsim_configs/togsim_config_timing_only.yml"
nonfusion_config = "/workspace/PyTorchSim/tutorial/session1/togsim_configs/togsim_config_no_compiler_optimization.yml"

### GeMM + ReLU fusion (Default)

In [2]:
os.environ['TORCHSIM_DUMP_PATH']=os.path.join(os.getcwd(), "fused")

device = torch.device("npu:0")

input = torch.randn(1024, 1024).to(device=device)
weight = torch.randn(1024, 1024).to(device=device)

def gemm_relu(a, b):
    return torch.relu(torch.matmul(a, b))

opt_fn = torch.compile(dynamic=False)(gemm_relu)
with TOGSimulator(config_path=fusion_config):
    npu_output = opt_fn(input, weight)

[2026-04-23 12:03:40.613] [info] [TOGSim] Command line: /workspace/PyTorchSim/TOGSim/build/bin/Simulator --config /workspace/PyTorchSim/tutorial/session1/togsim_configs/togsim_config_timing_only.yml --models_list /tmp/togsim_fifo_4831/cmd_fifo
[2026-04-23 12:03:40.614] [info] [LoadConfig] Loaded configuration file "/workspace/PyTorchSim/tutorial/session1/togsim_configs/togsim_config_timing_only.yml"
[2026-04-23 12:03:40.614] [info] PyTorchSim config:
num_cores: 1
core_freq_mhz: 940
core_stats_print_period_cycles: 10000
num_systolic_array_per_core: 2
vpu_num_lanes: 128
vpu_spad_size_kb_per_lane: 128
vpu_vector_length_bits: 256
dram_type: ramulator2
dram_freq_mhz: 940
dram_channels: 16
dram_stats_print_period_cycles: 10000
ramulator_config_path: /workspace/PyTorchSim/configs/ramulator2_configs/HBM2_TPUv3.yaml
icnt_type: simple
icnt_latency_cycles: 10
icnt_freq_mhz: 940
icnt_injection_ports_per_core: 16
pytorchsim_functional_mode: 0
pytorchsim_timing_mode: 1
codegen_mapping_strategy: heur

[2026-04-23 12:03:42.272] [INFO] [pytorchsimfrontend.mlir.generated_wrapper] Wrapper Codegen Path = /workspace/PyTorchSim/tutorial/session1/fused/.torchinductor/tn/ctnr7kcwozdwlge5i2b6vtqyf5fr42epo6q3brxznbs3vrahjqx4.py
[2026-04-23 12:03:42.435] [INFO] [simulator.simulator] [Gem5] Gem5 simulation started


[2026-04-23 12:03:56.476] [info] [LoadConfig] Loaded configuration file "/workspace/PyTorchSim/tutorial/session1/fused/yc6cmh6jmdj/runtime_0000/attribute/0"
[2026-04-23 12:03:56.479] [info] [Scheduler 0] Enqueued kernel_id: 0, tog_path: /workspace/PyTorchSim/tutorial/session1/fused/yc6cmh6jmdj/tile_graph.onnx, operation: relu_mm, request_time_cycles: 0
[2026-04-23 12:03:57.816] [info] ========= Core stat =========
[2026-04-23 12:03:57.816] [info] Core [0] : Systolic array [0] utilization(%): 29.41, active_cycles: 2941, idle_cycles: 7059
[2026-04-23 12:03:57.816] [info] Core [0] : Systolic array [1] utilization(%): 29.32, active_cycles: 2932, idle_cycles: 7068
[2026-04-23 12:03:57.816] [info] Core [0] : DMA active_cycles: 8899, DMA idle_cycles: 1101, DRAM BW: 294.000 GB/s (97926 responses)
[2026-04-23 12:03:57.816] [info] Core [0] : Vector unit Utilization(%): 10.97, active_cycles: 1097, idle_cycles: 8904
[2026-04-23 12:03:57.816] [info] Core [0] : Total_cycles: 10000
[2026-04-23 12:03:

[2026-04-23 12:04:01.159] [INFO] [simulator.simulator] [TOGSim] Trace log is stored to "/workspace/PyTorchSim/togsim_results/20260423_120401.trace"


[2026-04-23 12:04:01.151] [info] [Device 0] Device synchronization completed
[2026-04-23 12:04:01.151] [info] Simulation finished
[2026-04-23 12:04:01.151] [info] === DRAM statistics ===
--- channel 0 ---
frontend:
  impl: External

memory_system:
  impl: GenericDRAM
  total_num_read_requests: 16384
  total_num_write_requests: 8192
  channel_mapper:
    impl: PassThroughChannelMapper


  controller:
    impl: HBM
    id: Channel 0
    cycles: 53351
    row_hits: 18341
    row_misses: 321
    row_conflicts: 5914
    read_row_hits: 11192
    read_row_misses: 228
    read_row_conflicts: 4964
    write_row_hits: 7149
    write_row_misses: 93
    write_row_conflicts: 950
    read_row_hits_core_0: 11192
    read_row_misses_core_0: 228
    read_row_conflicts_core_0: 4964
    num_read_reqs: 16384
    num_write_reqs: 8192
    num_maintenance_reqs: 28
    num_read_reqs_served: 16384
    num_write_reqs_served: 8192
    num_maintenance_reqs_served: 28
    queue_len: 2010531
    read_queue_len: 143

### Disabling fusion

In [3]:
torch._dynamo.reset()
os.environ['TORCHSIM_DUMP_PATH']=os.path.join(os.getcwd(), "non_fused")
device = torch.device("npu:0")

input = torch.randn(1024, 1024).to(device=device)
weight = torch.randn(1024, 1024).to(device=device)

def gemm_relu(a, b):
    return torch.relu(torch.matmul(a, b))

opt_fn = torch.compile(dynamic=False)(gemm_relu)
with TOGSimulator(config_path=nonfusion_config):
    npu_output = opt_fn(input, weight)

[2026-04-23 12:04:01.196] [info] [TOGSim] Command line: /workspace/PyTorchSim/TOGSim/build/bin/Simulator --config /workspace/PyTorchSim/tutorial/session1/togsim_configs/togsim_config_no_compiler_optimization.yml --models_list /tmp/togsim_fifo_4831/cmd_fifo
[2026-04-23 12:04:01.197] [info] [LoadConfig] Loaded configuration file "/workspace/PyTorchSim/tutorial/session1/togsim_configs/togsim_config_no_compiler_optimization.yml"
[2026-04-23 12:04:01.197] [info] PyTorchSim config:
num_cores: 1
core_freq_mhz: 940
core_stats_print_period_cycles: 10000
num_systolic_array_per_core: 2
vpu_num_lanes: 128
vpu_spad_size_kb_per_lane: 128
vpu_vector_length_bits: 256
dram_type: ramulator2
dram_freq_mhz: 940
dram_channels: 16
dram_stats_print_period_cycles: 10000
ramulator_config_path: /workspace/PyTorchSim/configs/ramulator2_configs/HBM2_TPUv3.yaml
icnt_type: simple
icnt_latency_cycles: 10
icnt_freq_mhz: 940
icnt_injection_ports_per_core: 16
pytorchsim_functional_mode: 0
pytorchsim_timing_mode: 1
code

[2026-04-23 12:04:02.421] [INFO] [pytorchsimfrontend.mlir.generated_wrapper] Wrapper Codegen Path = /workspace/PyTorchSim/tutorial/session1/non_fused/.torchinductor/57/c574ab4fnwzq4ashxljqulic62n3kkmjz2cwiwegkdcq5fwfy3bn.py
[2026-04-23 12:04:02.551] [INFO] [simulator.simulator] [Gem5] Gem5 simulation started
[2026-04-23 12:04:02.590] [INFO] [simulator.simulator] [Gem5] Gem5 simulation started


[2026-04-23 12:04:08.443] [info] [LoadConfig] Loaded configuration file "/workspace/PyTorchSim/tutorial/session1/non_fused/p6df6jik75x/runtime_0000/attribute/0"
[2026-04-23 12:04:08.447] [info] [Scheduler 0] Enqueued kernel_id: 0, tog_path: /workspace/PyTorchSim/tutorial/session1/non_fused/p6df6jik75x/tile_graph.onnx, operation: mm, request_time_cycles: 0
[2026-04-23 12:04:08.454] [info] [LoadConfig] Loaded configuration file "/workspace/PyTorchSim/tutorial/session1/non_fused/ooqglyev4kb/runtime_0000/attribute/0"
[2026-04-23 12:04:08.466] [info] [Scheduler 0] Enqueued kernel_id: 1, tog_path: /workspace/PyTorchSim/tutorial/session1/non_fused/ooqglyev4kb/tile_graph.onnx, operation: relu, request_time_cycles: 0
[2026-04-23 12:04:09.798] [info] ========= Core stat =========
[2026-04-23 12:04:09.798] [info] Core [0] : Systolic array [0] utilization(%): 29.28, active_cycles: 2928, idle_cycles: 7072
[2026-04-23 12:04:09.798] [info] Core [0] : Systolic array [1] utilization(%): 29.19, active_c

[2026-04-23 12:04:14.159] [INFO] [simulator.simulator] [TOGSim] Trace log is stored to "/workspace/PyTorchSim/togsim_results/20260423_120414.trace"


[2026-04-23 12:04:14.105] [info] ========= Core stat =========
[2026-04-23 12:04:14.105] [info] Core [0] : Systolic array [0] utilization(%): 0.00, active_cycles: 0, idle_cycles: 10000
[2026-04-23 12:04:14.105] [info] Core [0] : Systolic array [1] utilization(%): 0.00, active_cycles: 0, idle_cycles: 10000
[2026-04-23 12:04:14.105] [info] Core [0] : DMA active_cycles: 1952, DMA idle_cycles: 8048, DRAM BW: 93.000 GB/s (31078 responses)
[2026-04-23 12:04:14.105] [info] Core [0] : Vector unit Utilization(%): 70.53, active_cycles: 7053, idle_cycles: 3191
[2026-04-23 12:04:14.105] [info] Core [0] : Total_cycles: 130000
[2026-04-23 12:04:14.105] [info] [DRAM] all 16 channels combined | 93.69 GB/s aggregate, 19.47% of utilization (avg. per channel) | 15561 reads, 15587 writes (interval 10000 cycles)
[2026-04-23 12:04:14.147] [info] [Scheduler 0] Kernel 1 has completed - TOG path: /workspace/PyTorchSim/tutorial/session1/non_fused/ooqglyev4kb/tile_graph.onnx operation: relu finished at cycle 132